In [1]:
import pandas as pd
import numpy as np

In [2]:
print(f"We need {15*20} profiles for labeling.")

We need 300 profiles for labeling.


In [7]:
def get_profile_url(account):
    return f"https://www.instagram.com/{account}/"

In [8]:
data_df = pd.read_csv("data_raw.csv", encoding="utf-8")
data_df = data_df[data_df["political"] == "political"]
data_df["profile_url"] = data_df["account"].apply(get_profile_url)
cleaned_df = data_df[["id", "account", "full_name", "biography",  "is_verified", "followers", "profile_url", "posts", "account_category", "political", "political_scale", "account_summary"]].copy()

In [9]:
print(cleaned_df.shape[0])
cleaned_df.head(1)

714


,id,account,full_name,biography,is_verified,followers,profile_url,posts,account_category,political,political_scale,account_summary
116,9.879575e+09,christopher_smith_nature,Christopher Smith,Information on the next steps in our fight aga...,False,12569,https://www.instagram.com/christopher_smith_na...,[{'caption': 'Thank you all so much for the su...,non-profit,political,moderate,The account focuses on sharing updates and adv...


In [10]:
cleaned_df["political_scale"].value_counts()

political_scale
moderate                  488
slightly liberal           52
extremely conservative     43
liberal                    43
conservative               30
extremely liberal          30
slightly conservative      27
non-applicable              1
Name: count, dtype: int64

In [11]:
students = [
    "sb132", "cd111", "ac163", "ct114", "os103",
    "rp104", "gb107", "ep123", "km124", "am149",
    "il115", "ec132", "ms162", "th118", "al141"
]

In [13]:
# Define exact quota per category per student
targets = {
    'moderate': 2,
    'slightly liberal': 2,
    'extremely conservative': 2,
    'liberal': 2,
    'conservative': 2,
    'extremely liberal': 2,
    'slightly conservative': 1
}

# Clean string formatting to avoid whitespace mismatches
cleaned_df['political_scale'] = cleaned_df['political_scale'].astype(str).str.strip().str.lower()
cleaned_df['user'] = np.nan

# Randomly select and assign posts for targeted categories only
for category, posts_per_student in targets.items():
    cat_indices = cleaned_df[cleaned_df['political_scale'] == category].index
    total_needed = posts_per_student * len(students)
    
    # Random sample without replacement
    sampled_indices = np.random.choice(cat_indices, size=total_needed, replace=False)
    
    # Assign equal chunks to each student
    for i, student in enumerate(students):
        start_idx = i * posts_per_student
        end_idx = start_idx + posts_per_student
        student_indices = sampled_indices[start_idx:end_idx]
        cleaned_df.loc[student_indices, 'user'] = student

# Drop rows that were not assigned to any student
cleaned_df = cleaned_df.dropna(subset=['user']).reset_index(drop=True)

C:\Users\chick\AppData\Local\Temp\ipykernel_26408\1908795880.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'sb132' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  cleaned_df.loc[student_indices, 'user'] = student


In [14]:
cleaned_df.to_csv("data_cleaned.csv", encoding="utf-8", index=False)